In [66]:
import pandas as pd
import geopandas as gpd 
from prometheo.models.presto.wrapper import (
    PretrainedPrestoWrapper,
    dataset_to_model,
    load_presto_weights,
)
from catboost import CatBoostClassifier
from pathlib import Path
import json 
from worldcereal_cop4geoglam.datasets import Cop4GeoLabelledDataset
from torch.utils.data import DataLoader
import numpy as np
import torch

In [81]:
def evaluate_presto(finetuned_model, test_ds, batch_size=64, num_workers=4, NODATAVALUE=65535, classes_list=None):
    finetuned_model.eval()
    # Construct the dataloader
    test_dl = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,  # keep as False!
        num_workers=num_workers,
    )

    # Run the model on the test set
    all_probs = []
    all_preds = []
    all_targets = []

    for batch in test_dl:
        with torch.no_grad():
            # batch may already be a Predictors or a dict collated by DataLoader
            if isinstance(batch, dict):
                batch = Predictors(**batch)

            model_output = finetuned_model(batch)
            targets = batch.label.cpu().numpy().astype(int)

            if test_ds.task_type == "binary":
                probs = torch.sigmoid(model_output).cpu().numpy()
                preds = (probs > 0.5).astype(int)
            elif test_ds.task_type == "multiclass":
                probs_all = (
                    torch.softmax(model_output, dim=-1).cpu().numpy()
                )  # shape (B,T,C)

                preds = np.argmax(probs_all, axis=-1, keepdims=True)
                probs = np.max(probs_all, axis=-1, keepdims=True)

                preds = preds[targets != NODATAVALUE]
                probs = probs[targets != NODATAVALUE]
                probs_all = probs_all[(targets != NODATAVALUE)[..., -1], :]
                targets = targets[targets != NODATAVALUE]
            else:
                raise ValueError(f"Unsupported task type: {test_ds.task_type}")

            all_probs.append(probs.flatten())
            all_preds.append(preds.flatten())
            all_targets.append(targets.flatten())

    all_probs = np.concatenate(all_probs)
    all_preds = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)

    # Map numeric indices to class names if necessary
    if test_ds.task_type == "multiclass" and classes_list:
        all_targets_classes = np.array(
            [classes_list[x] if x != NODATAVALUE else "unknown" for x in all_targets]
        )
        all_preds_classes = np.array([classes_list[x] for x in all_preds])

        # Remove any "unknown" targets before classification report
        valid_indices = all_targets_classes != "unknown"
        all_targets = list(all_targets_classes[valid_indices])
        all_preds = list(all_preds_classes[valid_indices])
        if len(all_probs) > 0:
            all_probs_array = np.array(all_probs)[valid_indices]
            all_probs = list(all_probs_array)
        else:
            all_probs = []
        classes_to_use = classes_list
    elif test_ds.task_type == "binary":
        # For binary classification, convert to class names
        all_targets = ["crop" if x > 0.5 else "not_crop" for x in all_targets]
        all_preds = list(
            np.array(["crop" if x > 0.5 else "not_crop" for x in all_preds])
        )
        classes_to_use = ["not_crop", "crop"]
    else:
        # Just use the classes as is
        classes_to_use = classes_list if classes_list is not None else []

    return all_targets, all_preds, all_probs, classes_to_use

def retrieve_ssu_id(sample_id):
 terms = sample_id.split('_')
 return '_'.join(terms[terms.index('POINT')+2:terms.index('POINT')+4])

In [82]:
root_path = Path('/vitodata/worldcereal/data/COP4GEOGLAM/mozambique/models/')
experiment_run_id = 'run=202509251303'
detector = 'croptype'
version = 1
dataset = "test" # "train", "val", "test"

In [83]:
presto_model_path = list((root_path / f'presto/v{version}/').glob(f'*{experiment_run_id}*'))[0]
presto_model_path = presto_model_path / f'{presto_model_path.name}_encoder.pt'
catboost_model_path = list((root_path / f'catboost/v{version}/{detector}').glob(f'*{experiment_run_id}*balance=True*'))[0]
catboost_model_path = catboost_model_path / f'{catboost_model_path.name}.cbm'

config_path = list(presto_model_path.parent.glob("config*.json"))[0]
with open(config_path, "r") as f:
    config = json.load(f)

In [84]:
# Load test data
test_df = pd.read_parquet(presto_model_path.parent / f'{dataset}_df.parquet')
test_ds = Cop4GeoLabelledDataset(
    test_df,
    num_timesteps=12,
    timestep_freq="month",
    task_type=config["task_type"],
    num_outputs=config["num_outputs"],
    time_explicit=False,
    classes_list=config["classes_list"],
    augment=False,  # No augmentation for testing
    label_jitter=0,  # No jittering for testing
    label_window=0,  # No windowing for testing
)

test_embeddings = pd.read_parquet(catboost_model_path.parent / f"{dataset}_embeddings.parquet")

In [85]:
# Load and evaluate Presto model
presto_model = PretrainedPrestoWrapper(
    num_outputs=num_outputs,
    regression=False,
)
presto_model = load_presto_weights(presto_model, presto_model_path, strict=False)
all_targets, all_preds, all_probs, classes_to_use = evaluate_presto(
    presto_model, 
    test_ds, 
    batch_size=64, 
    num_workers=2, 
    classes_list=config["classes_list"]
    )

In [86]:
# presto results 
presto_results = test_df.copy()
presto_results['predicted_class'] = all_preds
presto_results['target_class'] = all_targets
presto_results['id_ssu'] = presto_results['sample_id'].apply(lambda x: retrieve_ssu_id(x))
presto_results = presto_results[['sample_id', 'ewoc_code', 'finetune_class', 'balancing_class', 'predicted_class', 'target_class', 'id_ssu']]

In [97]:
# Load CatBoost model
catboost_model = CatBoostClassifier()
catboost_model.load_model(str(catboost_model_path))

# evaluate catboost 
emb_columns = [col for col in test_embeddings.columns if col.startswith('emb_')]
preds = catboost_model.predict(test_embeddings[emb_columns])
catboost_results = test_embeddings.copy()
catboost_results['predicted_class'] = preds.flatten()
catboost_results['id_ssu'] = catboost_results['sample_id'].apply(lambda x: retrieve_ssu_id(x))
catboost_results = catboost_results[['sample_id', 'ewoc_code', 'finetune_class', 'predicted_class', 'id_ssu']]